# Scaled dot-product attention from scratch in numpy

This notebook implements attention with numpy only and checks it against PyTorch:

```text
attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
```

For each query row, the output is a weighted average of the rows of `V`, with weights given by the softmax of the query's dot products with every key.

It matches these sections of notes.md:

- §3.1 (deriving attention): the formula above, implemented as `scaled_dot_product_attention`.
- §3.2 (why divide by $\sqrt{d_k}$): measured softmax entropy with and without the scaling at $d_k = 64$.
- §3.3 (worked example on 3 tokens): the same numbers, recomputed.
- §3.4 (multi-head attention): a `MultiHeadAttention` class, checked against `torch.nn.functional.scaled_dot_product_attention`.
- §3.7 (causal masking): the mask, and a check that no weight lands on future positions.

Run: `uv run jupyter execute ml/04_deep_learning/attention_scratch.ipynb`

## Setup

One random generator, `rng`, seeded with 42, supplies every random array below. The demos draw from it in order, so running the cells top to bottom reproduces the printed numbers exactly (notes §3.2 quotes the entropy values from this run).

In [1]:
import numpy as np

np.random.seed(42)
rng = np.random.default_rng(42)

## 1. Numerically stable softmax

Softmax is unchanged when the same constant is added to every logit, because the factor $e^c$ cancels between numerator and denominator. Subtracting the row maximum uses this: the largest exponent becomes $e^0 = 1$, so `exp` cannot overflow. The check below applies softmax to logits $(1000, 1001, 1002)$, which overflow a naive `exp`, and expects the same result as for $(0, 1, 2)$, about $(0.090, 0.245, 0.665)$.

In [2]:
def softmax(x, axis=-1):
    # Subtracting the max leaves the result unchanged (shift invariance)
    # and keeps exp() from overflowing.
    x = x - x.max(axis=axis, keepdims=True)  # shape preserved
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


big = softmax(np.array([1000.0, 1001.0, 1002.0]))
small = softmax(np.array([0.0, 1.0, 2.0]))
print("softmax(1000, 1001, 1002) =", np.round(big, 3))
print("softmax(   0,    1,    2) =", np.round(small, 3))
assert np.allclose(big, small)

softmax(1000, 1001, 1002) = [0.09  0.245 0.665]
softmax(   0,    1,    2) = [0.09  0.245 0.665]


## 2. Scaled dot-product attention, checked on the notes' worked example

`scaled_dot_product_attention` computes the score matrix $QK^\top/\sqrt{d_k}$, optionally sets scores above the diagonal to $-\infty$ (the causal mask of §3.7), applies a row-wise softmax, and multiplies by $V$. The cell then reruns the §3.3 example: three tokens $x_1 = (1,0)$, $x_2 = (0,1)$, $x_3 = (1,1)$ with $Q = K = V = X$. Expected: weight rows $(0.401, 0.198, 0.401)$, $(0.198, 0.401, 0.401)$, $(0.248, 0.248, 0.503)$, outputs $o_1 = (0.802, 0.599)$ and $o_3 = (0.752, 0.752)$, and with the causal mask a second row of $(0.330, 0.670, 0)$.

In [3]:
def scaled_dot_product_attention(Q, K, V, causal=False, scale=None):
    '''
    Q: (..., T_q, d_k)   queries, one row per output position
    K: (..., T_k, d_k)   keys, one row per input position (scored against Q)
    V: (..., T_k, d_v)   values, one row per input position (averaged)

    Returns:
      out: (..., T_q, d_v)   each row = weighted average of rows of V
      att: (..., T_q, T_k)   the attention weights (each row sums to 1)

    '...' is any batch shape, e.g. (B,) or (B, n_heads).
    '''
    d_k = Q.shape[-1]
    if scale is None:
        scale = 1.0 / np.sqrt(d_k)

    # scores[i, j] = <q_i, k_j> : similarity of query i to key j
    # (..., T_q, d_k) @ (..., d_k, T_k) -> (..., T_q, T_k)
    scores = (Q @ np.swapaxes(K, -1, -2)) * scale

    if causal:
        T_q, T_k = scores.shape[-2], scores.shape[-1]
        # mask[i, j] = True where j > i (the future). -inf logits -> weight 0.
        mask = np.triu(np.ones((T_q, T_k), dtype=bool), k=1)
        scores = np.where(mask, -np.inf, scores)

    att = softmax(scores, axis=-1)      # (..., T_q, T_k), rows sum to 1
    out = att @ V                       # (..., T_q, d_v)
    return out, att


# Notes §3.3: three tokens, d = 2, identity projections (Q = K = V = X).
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
out, att = scaled_dot_product_attention(X, X, X)
with np.printoptions(precision=3, suppress=True):
    print("attention weights:\n", att)
    print("outputs:\n", out)
assert np.allclose(att[0], [0.401, 0.198, 0.401], atol=1e-3)
assert np.allclose(att[2], [0.248, 0.248, 0.503], atol=1e-3)
assert np.allclose(out[0], [0.802, 0.599], atol=1e-3)
assert np.allclose(out[2], [0.752, 0.752], atol=1e-3)

out_c, att_c = scaled_dot_product_attention(X, X, X, causal=True)
with np.printoptions(precision=3, suppress=True):
    print("causal attention weights:\n", att_c)
assert np.allclose(att_c[0], [1.0, 0.0, 0.0])
assert np.allclose(att_c[1], [0.330, 0.670, 0.0], atol=1e-3)
assert np.allclose(att_c[2], att[2])   # the last row already saw every position

attention weights:
 [[0.401 0.198 0.401]
 [0.198 0.401 0.401]
 [0.248 0.248 0.503]]
outputs:
 [[0.802 0.599]
 [0.599 0.802]
 [0.752 0.752]]
causal attention weights:
 [[1.    0.    0.   ]
 [0.33  0.67  0.   ]
 [0.248 0.248 0.503]]


## 3. Shapes and row sums on random batched inputs

Draw a batch of 2 sequences, each with 5 positions and width 8, and run single-head attention on all of it at once (the leading batch axis passes through the matrix multiplies unchanged). Expected: the output has the shape of `V`, the weight matrix is $5 \times 5$ per sequence, and every row of weights sums to 1 up to floating-point error (around $10^{-16}$).

In [4]:
B, T, d_k, d_v = 2, 5, 8, 8
Q = rng.standard_normal((B, T, d_k))
K = rng.standard_normal((B, T, d_k))
V = rng.standard_normal((B, T, d_v))
out, att = scaled_dot_product_attention(Q, K, V)
print(f"Q {Q.shape}  K {K.shape}  V {V.shape}")
print(f"out {out.shape}   (expected ({B}, {T}, {d_v}))")
print(f"att {att.shape}   (expected ({B}, {T}, {T}))")
assert out.shape == (B, T, d_v) and att.shape == (B, T, T)
row_sums = att.sum(axis=-1)
print(f"attention rows sum to 1?  max |row_sum - 1| = "
      f"{np.abs(row_sums - 1).max():.2e}")
assert np.allclose(row_sums, 1.0)

Q (2, 5, 8)  K (2, 5, 8)  V (2, 5, 8)
out (2, 5, 8)   (expected (2, 5, 8))
att (2, 5, 5)   (expected (2, 5, 5))
attention rows sum to 1?  max |row_sum - 1| = 2.22e-16


## 4. Causal masking: no weight on the future

Rerun the same `Q, K, V` with `causal=True`. The printed matrix for the first sequence should be lower-triangular: row $i$ (a query) has nonzero weights only in columns $j \le i$. Row 0 can attend only to itself, so its single weight is exactly 1, and every entry above the diagonal is exactly 0 because $e^{-\infty} = 0$.

In [5]:
_, att_c = scaled_dot_product_attention(Q, K, V, causal=True)
print("attention matrix for batch 0 (rows=queries, cols=keys):")
with np.printoptions(precision=3, suppress=True):
    print(att_c[0])
upper = np.triu(np.ones((T, T), dtype=bool), k=1)
print(f"max weight above the diagonal: {att_c[:, upper].max():.2e} "
      f"(should be exactly 0)")
assert att_c[:, upper].max() == 0.0
# Row 0 can only attend to itself -> weight 1.0 on position 0.
assert np.allclose(att_c[:, 0, 0], 1.0)

attention matrix for batch 0 (rows=queries, cols=keys):
[[1.    0.    0.    0.    0.   ]
 [0.476 0.524 0.    0.    0.   ]
 [0.392 0.258 0.35  0.    0.   ]
 [0.188 0.103 0.346 0.363 0.   ]
 [0.173 0.205 0.219 0.251 0.152]]
max weight above the diagonal: 0.00e+00 (should be exactly 0)


## 5. Why divide by $\sqrt{d_k}$: softmax entropy at $d_k = 64$

If $q$ and $k$ have independent entries with mean 0 and variance 1, then $q^\top k$ is a sum of $d_k$ terms of variance 1, so its variance is $d_k$ (notes §3.2). At $d_k = 64$ the raw scores have standard deviation 8, and softmax of such spread-out scores puts almost all weight on one position. The cell measures the score variance before and after dividing by $\sqrt{d_k}$ (expected near 64 and near 1) and the mean entropy of the softmax rows over $T = 16$ keys. Entropy is $-\sum_j w_j \ln w_j$: 0 for a one-hot row and $\ln 16 \approx 2.77$ nats for a uniform row. Expected: about 0.46 nats unscaled and about 2.37 nats scaled.

In [6]:
def entropy(p, axis=-1):
    # Shannon entropy of each softmax row, in nats.
    # Max = ln(T) (uniform row), min = 0 (one-hot row).
    p = np.clip(p, 1e-12, 1.0)
    return -(p * np.log(p)).sum(axis=axis)


d = 64
Tn = 16
q = rng.standard_normal((Tn, d))
k = rng.standard_normal((Tn, d))
logits = q @ k.T                          # (T, T)
print(f"empirical Var(<q,k>) = {logits.var():7.2f}   (theory: d_k = {d})")
print(f"after /sqrt(d_k):      {(logits / np.sqrt(d)).var():7.2f}   (theory: 1)")
H_unscaled = entropy(softmax(logits)).mean()
H_scaled = entropy(softmax(logits / np.sqrt(d))).mean()
print(f"mean softmax entropy WITHOUT scaling: {H_unscaled:.3f} nats")
print(f"mean softmax entropy WITH    scaling: {H_scaled:.3f} nats")
print(f"max possible entropy ln({Tn}) =        {np.log(Tn):.3f} nats")
assert H_unscaled < 1.0 < H_scaled
print("Unscaled rows are close to one-hot, where the softmax Jacobian\n"
      "diag(w) - w w^T is close to 0, so gradients through the weights vanish.\n"
      "Scaled rows stay spread out.")

empirical Var(<q,k>) =   61.23   (theory: d_k = 64)
after /sqrt(d_k):         0.96   (theory: 1)
mean softmax entropy WITHOUT scaling: 0.456 nats
mean softmax entropy WITH    scaling: 2.368 nats
max possible entropy ln(16) =        2.773 nats
Unscaled rows are close to one-hot, where the softmax Jacobian
diag(w) - w w^T is close to 0, so gradients through the weights vanish.
Scaled rows stay spread out.


## 6. Multi-head attention

One head produces one attention distribution per query. Multi-head attention projects the input with $W_Q, W_K, W_V$, splits the $d_{\text{model}}$ columns into $h$ heads of width $d_{\text{model}}/h$, runs attention in each head independently, concatenates the head outputs, and multiplies by $W_O$ (notes §3.4). The cell builds a 4-head layer with $d_{\text{model}} = 16$ and runs it with the causal mask on a batch of 2 sequences of length 7. Expected shapes: output $(2, 7, 16)$, the same as the input, and attention weights $(2, 4, 7, 7)$, one $7 \times 7$ matrix per sequence and head.

In [7]:
class MultiHeadAttention:
    def __init__(self, d_model, n_heads, rng):
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        s = 1.0 / np.sqrt(d_model)  # small init; the scale does not matter here
        # In real implementations these are one fused (d_model, 3*d_model) matmul.
        self.Wq = rng.standard_normal((d_model, d_model)) * s  # (d, d)
        self.Wk = rng.standard_normal((d_model, d_model)) * s  # (d, d)
        self.Wv = rng.standard_normal((d_model, d_model)) * s  # (d, d)
        self.Wo = rng.standard_normal((d_model, d_model)) * s  # (d, d)

    def split_heads(self, x):
        # (B, T, d_model) -> (B, n_heads, T, d_head)
        B, T, _ = x.shape
        return x.reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)

    def __call__(self, x, causal=False):
        # x: (B, T, d_model)
        B, T, _ = x.shape
        Q = self.split_heads(x @ self.Wq)  # (B, h, T, d_head)
        K = self.split_heads(x @ self.Wk)  # (B, h, T, d_head)
        V = self.split_heads(x @ self.Wv)  # (B, h, T, d_head)

        out, att = scaled_dot_product_attention(Q, K, V, causal=causal)
        # out: (B, h, T, d_head) -> concat heads -> (B, T, d_model)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, self.d_model)
        return out @ self.Wo, att          # (B, T, d_model), (B, h, T, T)


B, T, d_model, n_heads = 2, 7, 16, 4
x = rng.standard_normal((B, T, d_model))
mha = MultiHeadAttention(d_model, n_heads, rng)
y, att = mha(x, causal=True)
print(f"x {x.shape} -> y {y.shape}   att {att.shape} (B, n_heads, T, T)")
assert y.shape == (B, T, d_model)
assert att.shape == (B, n_heads, T, T)

x (2, 7, 16) -> y (2, 7, 16)   att (2, 4, 7, 7) (B, n_heads, T, T)


## 7. Numerical check against PyTorch

Take the same projected and head-split $Q, K, V$ from the layer above, run the attention step through `torch.nn.functional.scaled_dot_product_attention` with `is_causal=True`, then apply the same head merge and $W_O$. If the numpy implementation is correct, the two outputs agree to floating-point precision: the expected maximum absolute difference is around $10^{-16}$, and the assert requires below $10^{-10}$.

In [8]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
# Same projections, same input; only the core attention runs in torch.
Qh = mha.split_heads(x @ mha.Wq)  # (B, h, T, d_head)
Kh = mha.split_heads(x @ mha.Wk)
Vh = mha.split_heads(x @ mha.Wv)
t_out = F.scaled_dot_product_attention(
    torch.from_numpy(Qh), torch.from_numpy(Kh), torch.from_numpy(Vh),
    is_causal=True,
).numpy()                                             # (B, h, T, d_head)
y_torch = t_out.transpose(0, 2, 1, 3).reshape(B, T, d_model) @ mha.Wo
max_err = np.abs(y - y_torch).max()
print(f"max |numpy - torch| = {max_err:.2e}")
assert max_err < 1e-10, "numpy and torch attention disagree!"
print("numpy multi-head attention matches torch.nn.functional."
      "scaled_dot_product_attention. All checks passed.")

max |numpy - torch| = 4.44e-16
numpy multi-head attention matches torch.nn.functional.scaled_dot_product_attention. All checks passed.
